In [ ]:
import json
import os
import spotipy
from spotipy.oauth2 import SpotifyOAuth
import boto3
import datetime

def lambda_handler(event, context):

    CLIENT_ID = os.environ["client_id"]
    CLIENT_SECRET = os.environ["client_secret"]
    REDIRECT_URI = os.environ["redirect_uri"]
    REFRESH_TOKEN = os.environ["refresh_token"]

    sp_oauth = SpotifyOAuth(
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET,
        redirect_uri=REDIRECT_URI,
        scope="playlist-read-private playlist-read-collaborative",
        open_browser=False
    )

    # Get a fresh access token using the refresh token
    token_info = sp_oauth.refresh_access_token(REFRESH_TOKEN)

    access_token = token_info["access_token"]

    # Create Spotify client
    sp = spotipy.Spotify(auth=access_token)

    playlist_id = "1npQQ8nEAzp256gIK6Mf1d"

    spotify_data = sp.playlist_tracks(playlist_id)

    print(spotify_data)

    client = boto3.client('s3')
    filename = 'spotify_raw_' + str(datetime.datetime.now()) + '.json'

    client.put_object(
        Bucket='spotify-etl-project-abhishek-anant', 
        Key='raw_data/to_be_processed/' + filename, 
        Body=json.dumps(spotify_data)
        )